In [ ]:
#PART 1 -     Cargar el dataset limpio generado en la Semana 2 en un DataFrame de Spark.
# INSTALACION DE PYSPARK

%pip install pyspark

print("Pyspark instalado -------------------")

Pyspark instalado -------------------


In [ ]:
# IMPORTS LIBRERIAS

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
import pandas as pd

print("librerias importadas -------------------------")

librerias importadas -------------------------


In [4]:
# CONFIGURACION DE SPARK - CREAR SESION

spark = SparkSession.builder\
    .master("local[*]")\
    .appName("DevOpsMetricsAnalytics_ProyectoFinal03")\
    .getOrCreate()

print(f"Spark versión: {spark.version}")
print(f"Paralelismo: {spark.sparkContext.defaultParallelism}")

Spark versión: 4.0.4
Paralelismo: 2


In [5]:
# PUNTO 1: DEFINIR RUTAS

BASE_DIR = Path("/content/")
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
DATA_ANALYTICS = BASE_DIR/"datos"/"analiticos"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_ANALYTICS.mkdir(parents=True, exist_ok=True)

print(f"Datos procesados: {DATA_PROCESSED}")
print(f"Datos analiticos: {DATA_ANALYTICS}")

Datos procesados: /content/datos/procesados
Datos analiticos: /content/datos/analiticos


In [6]:
# PUNTO 1: CARGAR DATASET LIMPIO DE LA GUIA 2 EN SPARK

ruta_entrada = DATA_PROCESSED/"dataset_limpio.csv"

print(f"Leyendo {ruta_entrada}")

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(ruta_entrada))

print("------------Dataset cargado en SPARK----------------")
print(f"FILAS: {df.count()}")
print(f"COLUMNAS: {len(df.columns)}")

Leyendo /content/datos/procesados/dataset_limpio.csv
------------Dataset cargado en SPARK----------------
FILAS: 20144
COLUMNAS: 25


In [8]:
# PUNTO 2: EXPLORACION BASICA

print("Esquema de datos: ")
df.printSchema()

print("\nPrimeras filas: =========")
df.show(truncate=False)

Esquema de datos: 
root
 |-- company_id: integer (nullable = true)
 |-- product_area: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- ticket_count: integer (nullable = true)
 |-- deployment_count: integer (nullable = true)
 |-- failed_deployments: integer (nullable = true)
 |-- avg_lead_time_hours: double (nullable = true)
 |-- rollback_count: integer (nullable = true)
 |-- incident_count: integer (nullable = true)
 |-- avg_resolution_time_hours: double (nullable = true)
 |-- total_downtime_min: integer (nullable = true)
 |-- avg_cpu_usage_pct: double (nullable = true)
 |-- avg_memory_usage_pct: double (nullable = true)
 |-- avg_response_time_ms: double (nullable = true)
 |-- avg_error_rate_pct: double (nullable = true)
 |-- avg_availability_pct: double (nullable = true)
 |-- avg_requests_per_minute: double (nullable = true)
 |-- deployment_failure_rate_pct: double (nullable = true)
 |-- rollback_rate_pct: double (nullable = true)
 |-- anio: integer (nullable = tr

In [9]:
# PARTICIONES (comprovacion)

print(f"Particiones iniciales: {df.rdd.getNumPartitions()}")

Particiones iniciales: 1


In [10]:
# PUNTO 3 (transformacion 1):  FILTER
# Decision: para analizar la tasa de fallos de deployment tiene sentido
# excluir los dias sin actividad de deployment, ya que incluirlos no aporta
# informacion sobre el desempeño de los deploys en si.

df_con_deploy = df.filter(F.col("deployment_count") > 0)

# Filtramos solo los dias donde SI hubo al menos un deployment.

print(f"Filas totales: {df.count()}")
print(f"Filas con deployment: {df_con_deploy.count()}")
print(f"Filas sin deployment (excluidas): {df.count() - df_con_deploy.count()}")

Filas totales: 20144
Filas con deployment: 10558
Filas sin deployment (excluidas): 9586


In [11]:
# PUNTO 3 (transformacion 2):   GROUPBY
# Agrupamos por area de producto

resumen_area = df.groupBy("product_area").agg(
    F.sum("ticket_count").alias("tickets_totales"),
    F.sum("deployment_count").alias("deployments_totales"),
    F.sum("incident_count").alias("incidentes_totales"),
    F.avg("avg_resolution_time_hours").alias("tiempo_resolucion_promedio")
)

print("Resumen por area de producto:")
resumen_area.show(truncate=False)

Resumen por area de producto:
+-------------+---------------+-------------------+------------------+--------------------------+
|product_area |tickets_totales|deployments_totales|incidentes_totales|tiempo_resolucion_promedio|
+-------------+---------------+-------------------+------------------+--------------------------+
|Data Pipeline|4885           |1423               |371               |6.301810089020774         |
|Notifications|9691           |2847               |739               |6.2867813567594           |
|Auth         |9707           |2868               |790               |6.1997896440129425        |
|Analytics    |9898           |2797               |761               |6.376508620689653         |
|Mobile       |10003          |2875               |776               |6.3304980420949635        |
|Billing      |5816           |2190               |563               |6.413275641025643         |
+-------------+---------------+-------------------+------------------+------------------

In [12]:
# PUNTO 3 (transformacion 3): ORDERBY
# Ordenar las areas de mayor a menor cantidad de incidentes,
# para identificar rapidamente que equipos tienen mas problemas operativos.

resumen_area_ordenado = resumen_area.orderBy(F.desc("incidentes_totales"))

print("Areas ordenadas por incidentes (mayor a menor):")
resumen_area_ordenado.show(truncate=False)

Areas ordenadas por incidentes (mayor a menor):
+-------------+---------------+-------------------+------------------+--------------------------+
|product_area |tickets_totales|deployments_totales|incidentes_totales|tiempo_resolucion_promedio|
+-------------+---------------+-------------------+------------------+--------------------------+
|Auth         |9707           |2868               |790               |6.1997896440129425        |
|Mobile       |10003          |2875               |776               |6.3304980420949635        |
|Analytics    |9898           |2797               |761               |6.376508620689653         |
|Notifications|9691           |2847               |739               |6.2867813567594           |
|Billing      |5816           |2190               |563               |6.413275641025643         |
|Data Pipeline|4885           |1423               |371               |6.301810089020774         |
+-------------+---------------+-------------------+------------------+

In [13]:
# PUNTO 4: CREAR VISTA TEMPORAL

df.createOrReplaceTempView("devops_metrics")
print("========== Vista temporal 'devops_metrics' creada ==========")

========== Vista temporal 'devops_metrics' creada ==========


In [14]:
# PUNTO 4: CONSULTA SQL
# Consultamos incidentes y tiempo de resolucion por area, replicando
# el mismo resultado que el groupBy anterior pero usando SQL puro,
# para demostrar que Spark permite ambos enfoques sobre los mismos datos.

consulta = """
  SELECT
    product_area,
    COUNT(*) AS dias_registrados,
    SUM(incident_count) AS incidentes_totales,
    ROUND(AVG(avg_resolution_time_hours), 2) AS tiempo_resolucion_promedio
  FROM devops_metrics
  GROUP BY product_area
  ORDER BY incidentes_totales DESC
"""

resultado_sql = spark.sql(consulta)
print("========== CONSULTA SQL - INCIDENTES POR AREA ==========")
resultado_sql.show(truncate=False)

========== CONSULTA SQL - INCIDENTES POR AREA ==========
+-------------+----------------+------------------+--------------------------+
|product_area |dias_registrados|incidentes_totales|tiempo_resolucion_promedio|
+-------------+----------------+------------------+--------------------------+
|Auth         |3884            |790               |6.2                       |
|Mobile       |3818            |776               |6.33                      |
|Analytics    |3762            |761               |6.38                      |
|Notifications|3917            |739               |6.29                      |
|Billing      |2812            |563               |6.41                      |
|Data Pipeline|1951            |371               |6.3                       |
+-------------+----------------+------------------+--------------------------+



In [15]:
# PUNTO 5: AGREGACION CON PANDAS (para comparar)
#importacion para calcular el tiempo de procesado de forma mas visual
import time

inicio_pandas = time.time()

df_pd = pd.read_csv(DATA_PROCESSED/"dataset_limpio.csv")
pd_area = df_pd.groupby("product_area")["incident_count"].sum().reset_index()
pd_area.columns = ["product_area", "incidentes_totales_pandas"]





fin_pandas = time.time()
tiempo_pandas = fin_pandas - inicio_pandas

print("========== PANDAS - INCIDENTES POR AREA ==========")
print(pd_area)
print(f"\nTiempo Pandas: {tiempo_pandas:.4f} segundos")

========== PANDAS - INCIDENTES POR AREA ==========
    product_area  incidentes_totales_pandas
0      Analytics                        761
1           Auth                        790
2        Billing                        563
3  Data Pipeline                        371
4         Mobile                        776
5  Notifications                        739

Tiempo Pandas: 0.1244 segundos


In [16]:
# PUNTO 5: AGREGACION CON SPARK (para comparar)

inicio_spark = time.time()

sp_area = df.groupBy("product_area").agg(
    F.sum("incident_count").alias("incidentes_totales_spark")
).toPandas()

fin_spark = time.time()
tiempo_spark = fin_spark - inicio_spark

print("========== SPARK - INCIDENTES POR AREA ==========")
print(sp_area)
print(f"\nTiempo Spark: {tiempo_spark:.4f} segundos")

========== SPARK - INCIDENTES POR AREA ==========
    product_area  incidentes_totales_spark
0  Data Pipeline                       371
1  Notifications                       739
2           Auth                       790
3      Analytics                       761
4         Mobile                       776
5        Billing                       563

Tiempo Spark: 0.8798 segundos


In [17]:
# COMPARAR TIEMPOS
# Recordatorio que con datasets pequeños (como este de 20k filas)
# pandas usualmente es mas rapido que Spark.

print(f"Tiempo Pandas: {tiempo_pandas:.4f}s")
print(f"Tiempo Spark: {tiempo_spark:.4f}s")


#print

if tiempo_pandas < tiempo_spark:
    print("\nPandas fue mas rapido en este dataset (esperado, por su tamaño pequeño)")
else:
    print("\nSpark fue mas rapido en este dataset")

Tiempo Pandas: 0.1244s
Tiempo Spark: 0.8798s

Pandas fue mas rapido en este dataset (esperado, por su tamaño pequeño)


In [18]:
# PUNTO 6: GUARDAR EN FORMATO PARQUET
# Parquet es un formato columnar optimizado para Big Data:
# se comprime mejor y es mas rapido de leer que CSV cuando se trabaja con Spark
# porque permite leer solo las columnas que necesitas.

ruta_parquet = DATA_ANALYTICS/"devops_metrics_parquet_equipo"

df.write \
    .mode("overwrite") \
    .parquet(str(ruta_parquet))

print(f"Parquet guardado en: {ruta_parquet}")

Parquet guardado en: /content/datos/analiticos/devops_metrics_parquet_equipo


In [19]:
# VERIFICAR: LEER EL PARQUET DE VUELTA

df_leido = spark.read.parquet(str(ruta_parquet))

print(f"Filas leidas: {df_leido.count()}")
print(f"Columnas leidas: {len(df_leido.columns)}")
df_leido.show(5, truncate=False)

Filas leidas: 20144
Columnas leidas: 25
+----------+------------+----------+------------+----------------+------------------+-------------------+--------------+--------------+-------------------------+------------------+-----------------+--------------------+--------------------+------------------+--------------------+-----------------------+---------------------------+-----------------+----+---+---+----------+-------------+-------------------+
|company_id|product_area|event_date|ticket_count|deployment_count|failed_deployments|avg_lead_time_hours|rollback_count|incident_count|avg_resolution_time_hours|total_downtime_min|avg_cpu_usage_pct|avg_memory_usage_pct|avg_response_time_ms|avg_error_rate_pct|avg_availability_pct|avg_requests_per_minute|deployment_failure_rate_pct|rollback_rate_pct|anio|mes|dia|dia_semana|es_fin_semana|sin_datos_monitoreo|
+----------+------------+----------+------------+----------------+------------------+-------------------+--------------+--------------+-------

## Punto 7: Ventajas y limitaciones encontradas — Spark vs Pandas

### Ventajas de Spark
- Permite dividir el dataset en particiones y procesar los datos en paralelo, en vez de secuencialmente como Pandas.
- Ofrece dos formas de trabajar sobre los mismos datos sin duplicar lógica: la API de DataFrame (`.groupBy()`, `.filter()`, `.orderBy()`) y consultas SQL puras (`spark.sql()`), ambas dieron resultados idénticos en nuestras pruebas.
- El formato Parquet (columnar) permite guardar y volver a leer grandes volúmenes de datos de forma más eficiente que CSV, ya que se pueden leer solo las columnas necesarias en vez de el archivo completo.
- Está diseñado para escalar a clústeres reales con múltiples máquinas, no solo a una sola computadora como Pandas.

### Limitaciones encontradas
- Con nuestro dataset de 20,144 filas, Spark fue notablemente más lento que Pandas (0.88s vs 0.12s) — casi 7 veces más tiempo.
- Esto se debe al costo fijo de coordinar el procesamiento distribuido (repartir los datos en particiones, coordinar el trabajo, juntar resultados), un costo que no se compensa con datasets pequeños.
- Configurar y entender Spark (sesión, particiones, planes de ejecución) tiene una curva de aprendizaje mayor que usar Pandas directamente.

### Conclusión
Para el tamaño actual de nuestro dataset de DevOps Metrics (~20K filas), Pandas es la herramienta más eficiente en este caso . Spark se justificaría si el proyecto creciera a escala real (múltiples empresas, monitoreo continuo por meses/años), donde el volumen de datos alcanzaría millones de filas y el procesamiento distribuido sí generaría una ventaja de rendimiento real y mas notable.